In [41]:
import numpy as np 
import pandas as pd 
import h5py
import re
import uproot
import glob 
import copy, sys, os
import matplotlib.pyplot as plt
from tqdm import tqdm 
# adding path to folder

from utils_analysis import ParticleCode, load_dataset
# Load your custom style 
plt.style.use('./cfg/my_custom_plot.mplstyle')

# use these lines on top of your matplotlib script
import matplotlib.ticker
class MyLocator(matplotlib.ticker.AutoMinorLocator):
    def __init__(self, n=4):
        super().__init__(n=n)
matplotlib.ticker.AutoMinorLocator = MyLocator        
 
# Now use matplotlib as usual.       
import matplotlib.pyplot as plt
plt.rcParams["xtick.minor.visible"] =  True
plt.rcParams["ytick.minor.visible"] =  True

sys.path.insert(0, '../tools/')
from caf_readers import CafReader
from mx2_matching import Mx2_DS_Match


In [42]:
# Load datasets 
n_files = 884
location = "nersc"
type="mr6"
df = load_dataset(n_files,location, type)
# Create a caf reader
caf_reader = CafReader(df)
pdg_tab = ParticleCode()

Reading  884  files gk
Location selected nersc
Openning MiniRun 6 CAFs


100%|██████████| 884/884 [49:27<00:00,  3.36s/it]


In [99]:
#dimensions of the detector
anode_xs = np.array([-63.931, -3.069, 3.069, 63.931])
anode_ys = np.array([-19.8543, 103.8543]) - 42  # Subtract 42 from all y coordinates
anode_zs = np.array([-64.3163, -2.6837, 2.6837, 64.3163])
#Fiducial Volume
tpc_dist = 5
xbound = 63.931
ybound = 62.076
zbound = 64.3163

def track_selection_signal(ev_data,ev):

    minerva_data = caf_reader.get_minerva_data(df.iloc[ev])

    x_start_track  = ev_data['rec.nd.lar.dlp.tracks.start.x']
    y_start_track = ev_data['rec.nd.lar.dlp.tracks.start.y']
    z_start_track = ev_data['rec.nd.lar.dlp.tracks.start.z']

    x_end_track = ev_data['rec.nd.lar.dlp.tracks.end.x']
    y_end_track = ev_data['rec.nd.lar.dlp.tracks.end.y']
    z_end_track = ev_data['rec.nd.lar.dlp.tracks.end.z']

    x_coords_vtx = ev_data['rec.common.ixn.dlp.vtx.x']
    y_coords_vtx = ev_data['rec.common.ixn.dlp.vtx.y']
    z_coords_vtx = ev_data['rec.common.ixn.dlp.vtx.z']

    # Stack the coordinates into a single 2D array
    position_vtx = np.vstack((x_coords_vtx, y_coords_vtx, z_coords_vtx)).T
    start_track = np.vstack((x_start_track, y_start_track, z_start_track)).T

    track_on_vtx = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)
    track_out = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)
    track_Mx2 = np.zeros_like(ev_data['rec.common.ixn.dlp.vtx.x'], dtype=bool)

    for vtx in range(len(ev_data['rec.common.ixn.dlp.vtx.x'])):
        n_reco_tracks = ev_data['rec.nd.lar.dlp.tracks..length'][vtx]
        n_pre = np.sum(ev_data['rec.nd.lar.dlp.tracks..length'][:vtx]) if vtx > 0 else 0
        
        vertex_position = position_vtx[vtx]
        # Scan over particles associated with this interaction
        found_match = False
        
        for ip in range(n_pre, n_pre + n_reco_tracks):
            if np.array_equal(vertex_position, start_track[ip]):
                found_match = True
                #if the track is found, then check if the track also goes out the boundary
                if (z_end_track[ip] >= zbound):
                    track_out[vtx] = True 
                    muon_track = [[x_start_track[ip], y_start_track[ip],z_start_track[ip]],[x_end_track[ip],y_end_track[ip],z_end_track[ip]]]
                    bm, overlap, dx, dy, exit_minerva = Mx2_DS_Match(muon_track,minerva_data)
                    if (exit_minerva == True) and (bm is not None):
                        track_Mx2[vtx] == True

        track_on_vtx[vtx] = found_match
    return track_on_vtx,  track_out, track_Mx2

In [173]:
ev_data=df.iloc[20]
n_ixn= len(ev_data['rec.common.ixn.dlp.vtx.z'])

print(f'This event has {n_ixn} reco vertices')
for ixn_index in range(n_ixn):
    print(f'Reco vertex {ixn_index}')
    n_int = ev_data['rec.common.ixn.dlp.truth..length'][ixn_index]
    if(ixn_index==0):
        n_pre = 0
    else: 
        n_pre = np.sum(ev_data['rec.common.ixn.dlp.truth..length'][:ixn_index]) 
    #print(n_pre)
    #print(n_pre,n_pre + n_int)
    max_overlap=0
    for ip in range(n_pre,n_pre + n_int):
        #print(ip)
        print(n_pre,n_pre + n_int  )
        temp_overlap = ev_data['rec.common.ixn.dlp.truthOverlap'][ip]
        temp_tp_ixn = ev_data['rec.common.ixn.dlp.truth'][ip]
        #print(f'this vtx has tem overlap {temp_overlap} in index {temp_tp_ixn}')
        if (temp_overlap> max_overlap):
            max_overlap = temp_overlap
            best_match_idx = temp_tp_ixn
        print(f'best overlap is at index {best_match_idx} with {max_overlap}')
# ev_data['rec.common.ixn.dlp.vtx.z']

This event has 1 reco vertices
Reco vertex 0
0 1
best overlap is at index 0 with 0.9756097793579102


In [181]:

num_neutrinos_fv_r=0 # in fv
num_neutrinos_fv_t=0 # in fv
num_neutrinos_fv_tt =0

#for tracks
num_track_vtx_out_fv=0 #produced track start on vtx and exit 2x2 downstream
num_track_Mx2_fv = 0 #num of tracks that crossed minerva 

count_reco=0
count_match_FV=0

filtered_data = []

for ev in range(len(df)):
   # print(ev)
    
    ev_data = df.iloc[ev]
    minerva_data = caf_reader.get_minerva_data(df.iloc[ev])
    

    #mask vertex on the fiducial volume
    mask_t = (
        (abs(ev_data['rec.mc.nu.vtx.x'])<xbound-tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.x'])>tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.y'])<ybound-tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.z'])>tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.z'])<zbound-tpc_dist) &
        (ev_data['rec.mc.nu.targetPDG']== pdg_tab.argon) &
        (ev_data['rec.mc.nu.iscc']==1) &
        (np.abs(ev_data['rec.mc.nu.pdg']) == pdg_tab.numu)
    )

      #mask vertex on the fiducial volume
    mask_tt = (
        (abs(ev_data['rec.mc.nu.vtx.x'])<xbound-tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.x'])>tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.y'])<ybound-tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.z'])>tpc_dist) &
        (abs(ev_data['rec.mc.nu.vtx.z'])<zbound-tpc_dist) 
    )
    #mask vertex on the fiducial volume
    mask_fv_r = (
        (abs(ev_data['rec.common.ixn.dlp.vtx.x'])<xbound-tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.x'])>tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.y'])<ybound-tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.z'])>tpc_dist) &
        (abs(ev_data['rec.common.ixn.dlp.vtx.z'])<zbound-tpc_dist)
    )
    
    num_neutrinos_fv_r += np.sum(mask_fv_r)
    num_neutrinos_fv_t += np.sum(mask_t)
    num_neutrinos_fv_tt += np.sum(mask_tt)

    #signal tracks in fv

    tracks_vtx, track_vtx_out, tracks_mx2 = track_selection_signal(ev_data,ev)
    tracks_vtx_fv = (tracks_vtx) & (mask_fv_r) #get them now in fv
    tracks_vtx_out_fv = (track_vtx_out) & (mask_fv_r)
    track_Mx2_fv = (tracks_mx2) & (mask_fv_r)
#    num_track_vtx_fv += np.sum(tracks_vtx_fv)
    num_track_vtx_out_fv+= np.sum(tracks_vtx_out_fv)
    num_track_Mx2_fv += np.sum(track_Mx2_fv)

    
    for ixn_index, value in enumerate(mask_fv_r):
        if value:  
            count_reco +=1 
            n_int = ev_data['rec.common.ixn.dlp.truth..length'][ixn_index]
            if(ixn_index==0):
                n_pre = 0
            else: 
                n_pre = np.sum(ev_data['rec.common.ixn.dlp.truth..length'][:ixn_index]) 
            max_overlap=0
            best_match_idx=None
            for ip in range(n_pre,n_pre + n_int):
                temp_overlap = ev_data['rec.common.ixn.dlp.truthOverlap'][ip]
                temp_tp_ixn = ev_data['rec.common.ixn.dlp.truth'][ip]
                #print(f'this vtx has tem overlap {temp_overlap} in index {temp_tp_ixn}')
                if (temp_overlap> max_overlap):
                    max_overlap = temp_overlap
                    best_match_idx = temp_tp_ixn

            if best_match_idx!=None:
                if mask_tt[best_match_idx]==True:
                    count_match_FV+=1
                    filtered_data.append(ev_data)
                    #print('contained')
        else: continue

                #print(f'best overlap is at index {best_match_idx} with {max_overlap}')
        

2x2 track length 134.86 cm
2x2 track length 135.24 cm
2x2 track length 13.70 cm
2x2 track length 9.80 cm
2x2 track length 129.45 cm
2x2 track length 129.73 cm
2x2 track length 130.07 cm
2x2 track length 129.30 cm
2x2 track length 129.98 cm
2x2 track length 135.13 cm
2x2 track length 129.73 cm
2x2 track length 91.57 cm
2x2 track length 130.83 cm
2x2 track length 129.15 cm
2x2 track length 120.80 cm
2x2 track length 134.11 cm
2x2 track length 129.76 cm
2x2 track length 61.90 cm
2x2 track length 91.47 cm
2x2 track length 129.13 cm
2x2 track length 3.57 cm
2x2 track length 129.58 cm
2x2 track length 25.53 cm
2x2 track length 129.43 cm
2x2 track length 129.09 cm
2x2 track length 129.45 cm
2x2 track length 84.36 cm
2x2 track length 129.06 cm
2x2 track length 130.10 cm
2x2 track length 132.78 cm
2x2 track length 15.33 cm
2x2 track length 134.44 cm
2x2 track length 25.26 cm
2x2 track length 129.30 cm
2x2 track length 58.94 cm
2x2 track length 129.51 cm
2x2 track length 62.79 cm
2x2 track lengt

/global/u2/g/gkufatty/projects/neutron-multiplicity/analysis/../tools/mx2_matching.py:75: RuntimeWarning: divide by zero encountered in scalar divide
  extrapdy = tpc_dy/tpc_dz*(extrapdz) + tpc_track_e[1] - mn_track_sy
/global/u2/g/gkufatty/projects/neutron-multiplicity/analysis/../tools/mx2_matching.py:76: RuntimeWarning: divide by zero encountered in scalar divide
  extrapdx = tpc_dx/tpc_dz*(extrapdz) + tpc_track_e[0] - mn_track_sx


2x2 track length 65.06 cm
2x2 track length 129.12 cm
2x2 track length 156.63 cm
2x2 track length 86.37 cm
2x2 track length 129.35 cm
2x2 track length 129.36 cm
2x2 track length 3.60 cm
2x2 track length 17.83 cm
2x2 track length 63.99 cm
2x2 track length 131.43 cm
2x2 track length 47.39 cm
2x2 track length 22.13 cm
2x2 track length 20.90 cm
2x2 track length 6.05 cm
2x2 track length 129.40 cm
2x2 track length 130.37 cm
2x2 track length 59.30 cm
2x2 track length 63.01 cm
2x2 track length 132.43 cm
2x2 track length 87.69 cm
2x2 track length 76.15 cm
2x2 track length 131.66 cm
2x2 track length 129.25 cm
2x2 track length 129.82 cm
2x2 track length 129.25 cm
2x2 track length 118.96 cm
2x2 track length 129.45 cm
2x2 track length 143.27 cm
2x2 track length 131.32 cm
2x2 track length 136.42 cm
2x2 track length 133.26 cm
2x2 track length 132.09 cm
2x2 track length 36.89 cm
2x2 track length 130.03 cm
2x2 track length 129.30 cm
2x2 track length 57.01 cm
2x2 track length 129.38 cm
2x2 track length 1

/global/u2/g/gkufatty/projects/neutron-multiplicity/analysis/../tools/mx2_matching.py:23: RuntimeWarning: invalid value encountered in divide
  tpc_dir_vec = np.array([tpc_dx,tpc_dy,tpc_dz])/tpc_trk_length


2x2 track length 130.13 cm
2x2 track length 131.44 cm
2x2 track length 157.38 cm
2x2 track length 130.22 cm
2x2 track length 132.58 cm
2x2 track length 129.56 cm
2x2 track length 131.34 cm
2x2 track length 78.52 cm
2x2 track length 131.76 cm
2x2 track length 62.67 cm
2x2 track length 57.69 cm
2x2 track length 129.40 cm
2x2 track length 129.31 cm
2x2 track length 129.62 cm
2x2 track length 129.92 cm
2x2 track length 129.49 cm
2x2 track length 129.31 cm
2x2 track length 130.47 cm
2x2 track length 129.18 cm
2x2 track length 129.82 cm
2x2 track length 129.84 cm
2x2 track length 129.18 cm
2x2 track length 40.94 cm
2x2 track length 129.47 cm
2x2 track length 129.04 cm
2x2 track length 129.25 cm
2x2 track length 129.52 cm
2x2 track length 78.93 cm
2x2 track length 130.54 cm
2x2 track length 9.42 cm
2x2 track length 129.30 cm
2x2 track length 62.67 cm
2x2 track length 44.20 cm
2x2 track length 6.71 cm
2x2 track length 131.48 cm
2x2 track length 129.15 cm
2x2 track length 130.76 cm
2x2 track le

In [182]:
print(f'there are {num_neutrinos_fv_r} reco vertices in FV, {num_neutrinos_fv_tt} interactions in FV, and {count_match_FV} vertices with true interaction match')

there are 61752 reco vertices in FV, 15812 interactions in FV, and 16749 vertices with true interaction match


In [174]:
def efficiency_purity(truth_int, reco_vtx, reco_vtx_matched):
    #for efficiency, we need the total num of interactions 
    # and the number we managed to reconstruct correctly 
    #for purity we need the real reco and the ones that were correctly reconstructed
    efficiency = (reco_vtx_matched/truth_int)*100
    purity = (reco_vtx_matched/reco_vtx)*100
    return efficiency, purity 


efficiency, purity = efficiency_purity(num_neutrinos_fv_t, num_track_vtx_out_fv,count_match_FV)    
print(efficiency, purity)

filtered_df = pd.DataFrame(filtered_data)



0.0 0.0
